In [2]:
from gen_auth_headers import gen_kalshi_auth_headers
import requests

In [3]:
ticker = "KXATPMATCH-26APR06VACCER-VAC"

kal_base_url = f"https://api.elections.kalshi.com/trade-api/v2"
kal_path = f"/markets/{ticker}/orderbook"

kal_params = {'status' : 'open', 'limit' : 200, 'with_nested_markets' : True}

auth_headers = gen_kalshi_auth_headers("GET", kal_path)


In [7]:
kal_resp = requests.get(kal_base_url + kal_path, headers=auth_headers)
kal_resp.raise_for_status()
kal_data = kal_resp.json()
kal_data

{'orderbook_fp': {'no_dollars': [['0.0100', '14150.00'],
   ['0.0200', '5264.00'],
   ['0.0300', '7248.00'],
   ['0.0400', '15187.00']],
  'yes_dollars': [['0.0100', '11600.00'],
   ['0.0200', '4801.00'],
   ['0.0300', '20800.00'],
   ['0.0400', '2408.00'],
   ['0.0500', '4170.00'],
   ['0.0600', '1800.00'],
   ['0.0700', '1200.00'],
   ['0.0800', '120.00'],
   ['0.1000', '11.00'],
   ['0.1100', '800.00'],
   ['0.1600', '3.00'],
   ['0.1700', '400.00'],
   ['0.2000', '2.00'],
   ['0.2100', '5.00'],
   ['0.2500', '1.00'],
   ['0.2600', '1.00'],
   ['0.2900', '20.00'],
   ['0.3100', '400.00'],
   ['0.3300', '2.00'],
   ['0.3400', '456.00'],
   ['0.3500', '1500.00'],
   ['0.3600', '41.00'],
   ['0.4500', '2.00'],
   ['0.4600', '1.00'],
   ['0.4700', '400.00'],
   ['0.5000', '1.00'],
   ['0.5100', '40.00'],
   ['0.5500', '254.00'],
   ['0.5600', '1.00'],
   ['0.6000', '2000.00'],
   ['0.6100', '7.00'],
   ['0.6700', '200.00'],
   ['0.7000', '2010.00'],
   ['0.7300', '12.00'],
   ['0.7500',

In [371]:
len(pol_events)

250

In [375]:
kal_titles = [e['title'] for e in kal_events]

In [376]:
pol_titles = [e['title'] for e in pol_events]

In [377]:
from sentence_transformers import SentenceTransformer

attn_implementation = "eager"
model_name = "all-mpnet-base-v2"

model = SentenceTransformer(
    model_name, trust_remote_code=True,
    model_kwargs={"attn_implementation": attn_implementation, "torch_dtype": "bfloat16"},
    tokenizer_kwargs={"padding_side": "left"},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
top_k = 3
thres = 0.8

for idx_pol, t_pol in enumerate(pol_titles):
    sims = torch.from_numpy(similarities[idx_pol])

    top_vals, top_idxs = torch.topk(sims, top_k)

    print(f'[{idx_pol}] {t_pol}')
    for score, idx_kal in zip(top_vals, top_idxs):
        if score > thres:
            print(f'- {score:.3f}: [{idx_kal}] {kal_titles[idx_kal]}')

[0] MicroStrategy sells any Bitcoin by ___ ?
[1] Kraken IPO by ___ ?
[2] Macron out by...?
[3] UK election called by...?
[4] China x India military clash by...?
[5] NATO/EU troops fighting in Ukraine by...?
[6] Starmer out by...?
[7] Ukraine recognizes Russian sovereignty over its territory by...?
[8] Ukraine election called by...?
[9] Will any country leave NATO by...?
[10] Ukraine election held by...?
[11] Taylor Swift pregnant in 2025?
[12] Mike Johnson out as Speaker by...?
[13] What will happen before GTA VI?
[14] GTA VI released before June 2026?
[15] Harvey Weinstein prison time?
[16] Will OpenAI launch a consumer hardware product by...?
[17] Will Russia capture Kostyantynivka by...?
[18] Spain snap election called by...?
[19] US x Russia military clash by...?
[20] Will Russia invade a NATO country by...?
[21] Trump eliminates capital gains tax on crypto by ___?
[22] Israel and Syria normalize relations by...?
[23] Will Russia capture Sumy by...?
[24] Pump.fun airdrop by ....? 


In [72]:
pol_idx = 72
kal_idx = 81

pol_markets = pol_resp[pol_idx]['markets']
kal_markets = kal_resp['events'][kal_idx]['markets']

# print(f'{pol_markets[0]['question']}\n\n{pol_markets[0]['description']}')
# print(f'{kal_markets[0]['title']}\n\n{kal_markets[0]['rules_primary']}\n\n{kal_markets[0]['rules_secondary']}')

pol_markets_desc = [f'{m['question']}\n\n{m['description']}' for m in pol_markets]
kal_markets_desc = [f'{m['title']}\n\n{m['rules_primary']}\n\n{m['rules_secondary']}' for m in kal_markets]

In [74]:
pol_mbdgs = model.encode(pol_markets_desc, normalize_embeddings=True)
kal_mbdgs = model.encode(kal_markets_desc, normalize_embeddings=True)

In [75]:
similarities = pol_mbdgs @ kal_mbdgs.T

In [78]:
top_k = 3
thres = 0.8

for idx_pol, t_pol in enumerate(pol_markets_desc):
    sims = torch.from_numpy(similarities[idx_pol])

    top_vals, top_idxs = torch.topk(sims, top_k)

    print(f'[{idx_pol}] {t_pol}')
    for score, idx_kal in zip(top_vals, top_idxs):
        if score > thres:
            print(f'- {score:.3f}: [{idx_kal}] {kal_markets_desc[idx_kal]}')

[0] Aaron Taylor-Johnson announced as next James Bond?

This market will resolve to “Yes” if the above actor is officially announced as the next James Bond actor by June 30, 2026, 11:59 PM ET. Otherwise, this market will resolve to “No”.

This market will resolve based on the first official announcement of who will be the next James Bond, regardless of any changes made thereafter.

If no actor is announced as the next Bond within the timeframe, this market will resolve to "No Bond chosen".

The primary resolution source for this market will be official information from Amazon MGM Studios. However, a consensus of credible reporting may also be used.
- 0.916: [1] Will Aaron Taylor-Johnson be the next James Bond?

If Aaron Taylor-Johnson is cast as the next James Bond before Jan 1, 2030, then the market resolves to Yes.

This is for the next film in the James Bond series, which goes back to Dr. No (1962); spin-offs and unauthorized productions are not included.
[1] No one announced as nex

In [379]:
def sim_search(collection1: list[str], collection2: list[str], k: int = 3, thres: float = 0.8):
    if not collection1 or not collection2:
        return {}

    k = min(k, len(collection2))
    
    col1_mbdgs = model.encode(collection1, normalize_embeddings=True)
    col2_mbdgs = model.encode(collection2, normalize_embeddings=True)
    
    similarities = col1_mbdgs @ col2_mbdgs.T

    topk_idx = np.argpartition(similarities, -k, axis=1)[:, -k:]
    topk_values = np.take_along_axis(similarities, topk_idx, axis=1)
    
    topk_thres_mask = topk_values >= thres

    collection1_res = np.where(topk_thres_mask.any(axis=1))[0]

    res = {}

    for idx in collection1_res:
        cur_idcs = topk_idx[idx][topk_thres_mask[idx]]
        cur_vals = topk_values[idx][topk_thres_mask[idx]]

        order = np.argsort(-cur_vals)

        res[int(idx)] = [
            (int(cur_idcs[j]), float(cur_vals[j]))
            for j in order
        ]

    return res

In [380]:
similar_events = sim_search(pol_titles, kal_titles, k=3, thres=0.75)

for pol_idx, sim in similar_events:
    


for idx_pol, t_pol in enumerate(pol_titles):
    sims = torch.from_numpy(similarities[idx_pol])

    top_vals, top_idxs = torch.topk(sims, top_k)

    print(f'[{idx_pol}] {t_pol}')
    for score, idx_kal in zip(top_vals, top_idxs):
        if score > thres:
            print(f'- {score:.3f}: [{idx_kal}] {kal_titles[idx_kal]}')

{25: [(34, 0.7791162133216858)],
 31: [(131, 0.7722987532615662)],
 33: [(227, 0.9087890386581421),
  (295, 0.8653604388237),
  (222, 0.7977206110954285)],
 35: [(108, 0.8890836238861084),
  (109, 0.837885856628418),
  (220, 0.7684178948402405)],
 37: [(226, 0.9100226759910583),
  (294, 0.8526852130889893),
  (221, 0.7909124493598938)],
 38: [(935, 0.7953401207923889), (384, 0.751861572265625)],
 39: [(468, 0.9062479138374329),
  (796, 0.8105846643447876),
  (926, 0.8069325089454651)],
 40: [(796, 0.9180270433425903),
  (469, 0.899750292301178),
  (797, 0.8938632607460022)],
 42: [(467, 0.8793174624443054)],
 44: [(425, 0.7962403297424316)],
 47: [(109, 0.9079693555831909),
  (100, 0.8331207036972046),
  (108, 0.8222435712814331)],
 57: [(391, 0.9144524931907654)],
 63: [(143, 0.8519458174705505), (120, 0.8355206251144409)],
 64: [(105, 0.8636924624443054)],
 66: [(381, 0.8525967001914978),
  (253, 0.785163164138794),
  (252, 0.7661327719688416)],
 67: [(381, 0.7725666761398315)],
 68:

In [ ]:
for pol_idx, matches in similar_events.items():
    pol_markets = pol_events[pol_idx]['markets']
    kal_markets = [m for (kal_idx, _) in matches for m in kal_events[kal_idx]['markets']]

    pol_markets_desc = [f"{m['question']}\n\n{m['description']}" for m in pol_markets]
    
    kal_market_records = []
    for kal_idx, event_score in matches:
        event = kal_events[kal_idx]
        for market_idx, market in enumerate(event['markets']):
            kal_market_records.append({
                'event_idx': kal_idx,
                'event_title': event['title'],
                'event_score': event_score,
                'market_idx': market_idx,
                'title': market.get('title', ''),
                'rules_primary': market.get('rules_primary', ''),
                'rules_secondary': market.get('rules_secondary', ''),
            })
    
    kal_markets_desc = [
        f"{r['title']}\n\n{r['rules_primary']}\n\n{r['rules_secondary']}"
        for r in kal_market_records
    ]

    similar_markets = sim_search(pol_markets_desc, kal_markets_desc, k=3, thres=0.5)

    for pol_m_idx, kal_sim in similar_markets.items():
        print("=" * 30)
        print(f'[{pol_idx}][{pol_m_idx}] {pol_markets_desc[pol_m_idx]}')
        for (kal_m_idx, score) in kal_sim:
            print("-" * 30)
            print(f'[{kal_market_records[kal_m_idx]['event_idx']}][{kal_market_records[kal_m_idx]['market_idx']}] {score:.3f} {kal_markets_desc[kal_m_idx]}')

[25][0] Will the Carolina Hurricanes win the 2026 NHL Stanley Cup?

This market will resolve to “Yes” if the Carolina Hurricanes win the 2026 NHL Stanley Cup. Otherwise, this market will resolve to “No”.

This market will resolve to “No” if it becomes impossible for this team to win the 2026 NHL Stanley Cup based off the rules of the NHL.

The resolution source for this market will be information from the NHL.

------------------------------
[34][0] 0.772 Will a Canadian team win the pro hockey championship by the end of the 2030 season?

If a Canadian hockey team wins the pro hockey championship before Jun 30, 2030, then the market resolves to Yes.


[25][1] Will the Dallas Stars win the 2026 NHL Stanley Cup?

This market will resolve to “Yes” if the Dallas Stars win the 2026 NHL Stanley Cup. Otherwise, this market will resolve to “No”.

This market will resolve to “No” if it becomes impossible for this team to win the 2026 NHL Stanley Cup based off the rules of the NHL.

The resoluti

In [315]:
pol_titles[72], kal_titles[81]

('Next James Bond actor?', 'Who will be the next James Bond?')

In [347]:
kal_resp['events'][108]['markets'][12]

{'can_close_early': True,
 'close_time': '2029-11-07T15:00:00Z',
 'created_time': '2025-05-10T13:18:37.659307Z',
 'custom_strike': {'Pres candidate': 'Donald J. Trump'},
 'early_close_condition': 'This market will close and expire after a person has been inaugurated as President pursuant to the next presidential election.',
 'event_ticker': 'KXPRESPERSON-28',
 'expected_expiration_time': '2029-01-21T15:00:00Z',
 'expiration_time': '2029-11-07T15:00:00Z',
 'expiration_value': '',
 'fractional_trading_enabled': False,
 'last_price_dollars': '0.0220',
 'latest_expiration_time': '2029-11-07T15:00:00Z',
 'liquidity_dollars': '0.0000',
 'market_type': 'binary',
 'no_ask_dollars': '0.9800',
 'no_bid_dollars': '0.9780',
 'no_sub_title': 'Donald J. Trump',
 'notional_value_dollars': '1.0000',
 'open_interest_fp': '1258072.00',
 'open_time': '2025-05-10T14:00:00Z',
 'previous_price_dollars': '0.0220',
 'previous_yes_ask_dollars': '0.0220',
 'previous_yes_bid_dollars': '0.0200',
 'price_level_str